## Lung Disease Dataset

Author: Behnam Asadi



#### ARCHITECTURE EXPLANATION:

For this project I use a single backbone: `tf_efficientnetv2_l` from the `timm` library, initialized with ImageNet pretrained weights. Chest X-ray images are first converted to 3-channel RGB, resized to 384×384, and normalized using pre-computed dataset statistics with mean $\mu = [0.4949, 0.4949, 0.4949]$ and standard deviation $\sigma = [0.2447, 0.2447, 0.2447]$, before being passed through the EfficientNetV2-L convolutional backbone to produce a 1,280-dimensional feature vector.

I replace the original classifier head with a custom medical-specific head:
- Global adaptive average pooling over the feature map, followed by flattening
- `LayerNorm` on the 1,280-dim feature vector
- `Dropout(0.3)` for regularization
- `Linear(1280 → 128)` + `ReLU` non-linearity
- `Dropout(0.25)`
- `Linear(128 → 5)` to output logits for the five lung-disease classes

Training is done in two stages. In **Stage 1** I freeze the EfficientNetV2 backbone and train only the new head for 5 epochs with Adam (learning rate $1\times10^{-4}$). In **Stage 2** I unfreeze the whole network and fine-tune with AdamW using different learning rates for backbone and head (head LR $3\times10^{-5}$, backbone LR $5\times10^{-6}$), together with a cosine learning-rate schedule and label-smoothed cross-entropy loss (label smoothing $\epsilon = 0.1$).

Data augmentation includes random horizontal flips, small rotations (±10°), and mild color jitter, which helps the model generalize to variations in acquisition conditions. This combination of a strong pretrained backbone, regularized classification head, and conservative two-stage fine-tuning gives the best trade-off between accuracy and robustness on the Lung Disease Dataset.


### Solution overview

This notebook implements a **single strong baseline** for the Kaggle *Lungs Disease Dataset* using `tf_efficientnetv2_l` from the `timm` library.
The goal is to provide a **clear, well-documented reference solution** that other competitors can easily understand, reuse, and extend.

**Pipeline summary**
- **1. Model**: `tf_efficientnetv2_l` pretrained on ImageNet, with a custom medical head:
  global average pooling → LayerNorm → Dropout(0.3) → Linear(1280→128) → ReLU → Dropout(0.25) → Linear(128→5).
- **2. Data**: images are loaded from pre-split `train/val/test` folders (no splitting in the notebook), converted to RGB, resized to 384×384 and normalized with dataset-specific mean/std computed offline.
- **3. Augmentation**: horizontal flip, small rotations (±10°), and light color jitter, which are standard but conservative choices for chest X-ray images.
- **4. Optimization**: full fine-tuning with AdamW, **different learning rates** for backbone and head (`5e-6` vs `3e-5`), label smoothing (`ε = 0.1`), and `ReduceLROnPlateau` scheduling with early stopping.
- **5. Evaluation**: accuracy, detailed classification report, and a confusion matrix matching the paper-style plots from the main project.

You can scroll through the notebook to see the implementation in four logical blocks:
1. **Model definition & hyperparameters**
2. **Setup, dataset download, and preprocessing**
3. **Training & evaluation utilities**
4. **`run_lung_disease_experiment()` end-to-end entry point** (one function call to reproduce the full experiment on Kaggle GPU).


### Notebook overview

This notebook provides a complete **end-to-end solution** for the *Lung Disease Dataset* on Kaggle. The goal is to classify chest X‑ray images into five categories (Bacterial Pneumonia, Corona Virus Disease, Normal, Tuberculosis, Viral Pneumonia) using a strong modern backbone (`tf_efficientnetv2_l`) with a custom medical head.

The notebook is intentionally **self‑contained** so other Kaggle users can easily:
- Understand the **architecture and design choices** (with medical‑oriented explanations).
- **Reproduce training** on the public Kaggle GPU with a single `run_lung_disease_experiment()` call.
- Inspect **evaluation metrics**, confusion matrix, and training curves for further analysis.

The main sections are:
1. **Architecture explanation** (high‑level description of the model and training strategy).
2. **Model implementation** – EfficientNetV2‑L backbone + custom classification head.
3. **Data loading & preprocessing** – KaggleHub download + ImageFolder on pre‑split `train/val/test` folders.
4. **Training & evaluation utilities** – optimizer, scheduler, early stopping, metrics, and plots.
5. **Main experiment runner** – a single function that ties everything together for easy reuse.


In [35]:
!pip install -q timm > /dev/null 2>&1

In [36]:
import timm
import torch.nn as nn

# Hyperparameters and model settings (taken directly from your training config)
MODEL_NAME = "tf_efficientnetv2_l"
NUM_CLASSES = 5
INPUT_SIZE = 384         # 384 × 384 input resolution for EfficientNetV2-L
#BATCH_SIZE = 8           # From train_runpod.yaml / model_config
BATCH_SIZE = 1           # From train_runpod.yaml / model_config
STAGE1_EPOCHS = 5
STAGE2_EPOCHS = 15
HEAD_LR = 3e-5           # training.stage2.head_lr
BACKBONE_LR = 5e-6       # training.stage2.backbone_lr
LABEL_SMOOTHING = 0.1    # loss.label_smoothing


class LungDiseaseEfficientNetV2L(nn.Module):
    """EfficientNetV2-L backbone with the custom medical head used in your project.

    Classifier head:
        AdaptiveAvgPool2d(1) → Flatten → LayerNorm → Dropout(0.3)
        → Linear(num_features → 128) → ReLU → Dropout(0.25) → Linear(128 → NUM_CLASSES)
    """

    def __init__(self, pretrained: bool = True):
        super().__init__()

        # Create EfficientNetV2-L backbone from timm
        model = timm.create_model(MODEL_NAME, pretrained=pretrained)
        num_features = model.num_features

        # Replace the original head with our custom medical classifier
        custom_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),   # [B, C, H, W] -> [B, C, 1, 1]
            nn.Flatten(1),             # [B, C, 1, 1] -> [B, C]
            nn.LayerNorm(num_features),
            nn.Dropout(0.3),
            nn.Linear(num_features, 128),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(128, NUM_CLASSES),
        )

        model.head = custom_head
        self.model = model

    def forward(self, x):
        return self.model(x)



In [37]:
import os
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import torchvision.transforms as transforms
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

import kagglehub

# Dataset-specific normalization constants (from project `data.yaml`)
NORMALIZATION_MEAN = [0.49495694, 0.49495694, 0.49495694]
NORMALIZATION_STD = [0.24466455, 0.24466455, 0.24466455]

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if device.type == "cuda":
    torch.cuda.manual_seed_all(SEED)


def download_lungs_dataset():
    """Download dataset and find train/val/test directories.

    Uses the public Kaggle dataset: omkarmanohardalvi/lungs-disease-dataset-4-types
    """
    # Download dataset
    dataset_path = kagglehub.dataset_download("omkarmanohardalvi/lungs-disease-dataset-4-types")
    print(f"✅ Dataset downloaded to: {dataset_path}")
    
    dataset_path = Path(dataset_path)
    
    # Look for train/val/test directories
    possible_train_dirs = list(dataset_path.rglob("train"))
    possible_val_dirs = list(dataset_path.rglob("val"))
    possible_test_dirs = list(dataset_path.rglob("test"))
    
    train_path = possible_train_dirs[0] if possible_train_dirs else None
    val_path = possible_val_dirs[0] if possible_val_dirs else None
    test_path = possible_test_dirs[0] if possible_test_dirs else None
    
    if train_path and val_path:
        print(f"✅ Found train: {train_path}")
        print(f"✅ Found val: {val_path}")
        if test_path:
            print(f"✅ Found test: {test_path}")
        return str(train_path), str(val_path), str(test_path) if test_path else None
    else:
        raise RuntimeError(f"Could not find train/val directories in {dataset_path}")


Using device: cuda


In [38]:
def get_transforms(image_size: int = INPUT_SIZE):
    """Augmentation and preprocessing for lung X-ray images."""
    # Train transforms with augmentation
    train_tfm = transforms.Compose([
        transforms.Lambda(lambda img: img.convert("RGB")),  # Convert to RGB
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(0.2, 0.2, 0.2, 0),
        transforms.ToTensor(),
        transforms.Normalize(mean=NORMALIZATION_MEAN, std=NORMALIZATION_STD),
    ])
    
    # Val/test transforms (no augmentation)
    val_tfm = transforms.Compose([
        transforms.Lambda(lambda img: img.convert("RGB")),
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=NORMALIZATION_MEAN, std=NORMALIZATION_STD),
    ])
    return train_tfm, val_tfm


def build_datasets(train_path: str, val_path: str, test_path: str | None):
    """Build datasets using ImageFolder directly (no splitting."""
    train_tfm, val_tfm = get_transforms(image_size=INPUT_SIZE)
    
    # Use ImageFolder directly with pre-split directories
    train_ds = ImageFolder(train_path, transform=train_tfm)
    val_ds = ImageFolder(val_path, transform=val_tfm)
    test_ds = ImageFolder(test_path, transform=val_tfm) if test_path else None
    
    # Get class names from train dataset
    classes = train_ds.classes
    
    print("Dataset sizes:")
    print(f"  Train: {len(train_ds)} samples")
    print(f"  Val:   {len(val_ds)} samples")
    if test_ds:
        print(f"  Test:  {len(test_ds)} samples")
    print(f"  Classes: {classes}")
    
    return train_ds, val_ds, test_ds, classes


def build_loaders(train_ds, val_ds, test_ds, batch_size: int = BATCH_SIZE, num_workers: int = 4):
    """Build DataLoaders."""
    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=(device.type == "cuda"),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=(device.type == "cuda"),
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=(device.type == "cuda"),
    ) if test_ds else None
    return train_loader, val_loader, test_loader


In [39]:
def build_optimizer(model: nn.Module):
    """AdamW optimizer with different LRs for backbone and head.

    Uses HEAD_LR and BACKBONE_LR constants from this notebook.
    """
    head_params = []
    backbone_params = []

    if hasattr(model, "model") and hasattr(model.model, "head"):
        head_param_ids = {id(p) for p in model.model.head.parameters()}
        for p in model.parameters():
            if id(p) in head_param_ids:
                head_params.append(p)
            else:
                backbone_params.append(p)
    else:
        # Fallback: all parameters use the same LR
        return optim.AdamW(model.parameters(), lr=HEAD_LR, weight_decay=0.01)

    param_groups = []
    if backbone_params:
        param_groups.append({"params": backbone_params, "lr": BACKBONE_LR, "weight_decay": 0.01})
    if head_params:
        param_groups.append({"params": head_params, "lr": HEAD_LR, "weight_decay": 0.01})

    print(f"Optimizer: AdamW (backbone lr={BACKBONE_LR:.1e}, head lr={HEAD_LR:.1e})")
    return optim.AdamW(param_groups)


def train_one_epoch(model, loader, device, criterion, optimizer):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, targets in tqdm(loader, desc="Train", leave=False):
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)

    return total_loss / total, correct / total


def evaluate(model, loader, device, criterion):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for images, targets in tqdm(loader, desc="Eval", leave=False):
            images, targets = images.to(device), targets.to(device)
            outputs = model(images)
            loss = criterion(outputs, targets)

            total_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == targets).sum().item()
            total += targets.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())

    return total_loss / total, correct / total, np.array(all_preds), np.array(all_targets)


def train_model(model, train_loader, val_loader, epochs: int = STAGE1_EPOCHS + STAGE2_EPOCHS, patience: int = 5):
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING).to(device)
    optimizer = build_optimizer(model)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_acc = 0.0
    best_state = None
    patience_counter = 0

    for epoch in range(1, epochs + 1):
        print(f"Epoch {epoch}/{epochs}")
        train_loss, train_acc = train_one_epoch(model, train_loader, device, criterion, optimizer)
        val_loss, val_acc, _, _ = evaluate(model, val_loader, device, criterion)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        scheduler.step(val_acc)

        print(
            f"  Train: loss={train_loss:.4f}, acc={train_acc*100:.2f}% | "
            f"Val: loss={val_loss:.4f}, acc={val_acc*100:.2f}%"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = model.state_dict()
            patience_counter = 0
            print(f"  🔥 New best val acc: {best_val_acc*100:.2f}%")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("  Early stopping triggered.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return history, best_val_acc


def plot_training(history):
    epochs = len(history["train_loss"])
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].plot(range(1, epochs + 1), history["train_loss"], label="Train")
    axes[0].plot(range(1, epochs + 1), history["val_loss"], label="Val")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("Loss")
    axes[0].legend()

    axes[1].plot(range(1, epochs + 1), np.array(history["train_acc"]) * 100, label="Train")
    axes[1].plot(range(1, epochs + 1), np.array(history["val_acc"]) * 100, label="Val")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy (%)")
    axes[1].set_title("Accuracy")
    axes[1].legend()

    plt.tight_layout()
    plt.show()


def plot_confusion_matrix(y_true, y_pred, classes):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix - Lung Disease Classification")
    plt.tight_layout()
    plt.show()


In [40]:
def run_lung_disease_experiment():
    """Main experiment function."""

    # 1) Download dataset and get train/val/test paths (already split)
    train_path, val_path, test_path = download_lungs_dataset()
    
    # 2) Build datasets using ImageFolder directly (no splitting)
    train_ds, val_ds, test_ds, classes = build_datasets(train_path, val_path, test_path)
    train_loader, val_loader, test_loader = build_loaders(train_ds, val_ds, test_ds, batch_size=BATCH_SIZE)

    # 3) Build model
    model = LungDiseaseEfficientNetV2L(pretrained=True).to(device)
    print(f"Model has {sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable parameters")

    # 4) Train
    history, best_val_acc = train_model(
        model, train_loader, val_loader,
        epochs=STAGE1_EPOCHS + STAGE2_EPOCHS,
        patience=5,
    )
    print(f"Best validation accuracy: {best_val_acc*100:.2f}%")

    # 5) Evaluate on test set (if available)
    if test_loader:
        criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING).to(device)
        test_loss, test_acc, y_pred, y_true = evaluate(model, test_loader, device, criterion)
        print(f"Test loss: {test_loss:.4f}, Test accuracy: {test_acc*100:.2f}%")
        print("\nClassification report:\n")
        print(classification_report(y_true, y_pred, target_names=classes))
        
        # Visualizations
        plot_training(history)
        plot_confusion_matrix(y_true, y_pred, classes)
    else:
        print("⚠️  No test set available, skipping test evaluation")
        plot_training(history)


# To run end-to-end training + evaluation on Kaggle GPU, execute:
run_lung_disease_experiment()


✅ Dataset downloaded to: /home/behnam/.cache/kagglehub/datasets/omkarmanohardalvi/lungs-disease-dataset-4-types/versions/1
✅ Found train: /home/behnam/.cache/kagglehub/datasets/omkarmanohardalvi/lungs-disease-dataset-4-types/versions/1/Lung Disease Dataset/train
✅ Found val: /home/behnam/.cache/kagglehub/datasets/omkarmanohardalvi/lungs-disease-dataset-4-types/versions/1/Lung Disease Dataset/val
✅ Found test: /home/behnam/.cache/kagglehub/datasets/omkarmanohardalvi/lungs-disease-dataset-4-types/versions/1/Lung Disease Dataset/test
Dataset sizes:
  Train: 6054 samples
  Val:   2016 samples
  Test:  2025 samples
  Classes: ['Bacterial Pneumonia', 'Corona Virus Disease', 'Normal', 'Tuberculosis', 'Viral Pneumonia']


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.00 MiB. GPU 0 has a total capacity of 3.68 GiB of which 3.62 MiB is free. Including non-PyTorch memory, this process has 3.66 GiB memory in use. Of the allocated memory 3.51 GiB is allocated by PyTorch, and 55.42 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)